# Load and Setup Video Processing
Import required libraries (cv2, os) and set up basic video processing parameters like fps and codec.

In [1]:
# Import required libraries
import cv2
import os

# Set up basic video processing parameters
fps = 30  # Frames per second
codec = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for MP4 format

# Function to load a video file
def load_video(video_path):
    if not os.path.exists(video_path):
        raise FileNotFoundError(f"Video file {video_path} not found.")
    return cv2.VideoCapture(video_path)

# Function to set up video writer
def setup_video_writer(output_path, frame_width, frame_height):
    return cv2.VideoWriter(output_path, codec, fps, (frame_width, frame_height))

# Define Video Cutting Functions
Create functions to handle video cutting operations using both timestamp and frame number approaches.

In [2]:
# Define Video Cutting Functions

def cut_video_by_time(input_path, output_path, start_time, end_time):
    """
    Cut video from start_time to end_time and save as a new file.
    
    Parameters:
    - input_path: Path to the input video file.
    - output_path: Path to save the cut video.
    - start_time: Start time in seconds.
    - end_time: End time in seconds.
    """
    cap = load_video(input_path)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out = setup_video_writer(output_path, frame_width, frame_height)
    
    start_frame = int(start_time * fps)
    end_frame = int(end_time * fps)
    
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    
    for frame_num in range(start_frame, end_frame):
        ret, frame = cap.read()
        if not ret:
            break
        out.write(frame)
    
    cap.release()
    out.release()

def cut_video_by_frame(input_path, output_path, start_frame, end_frame):
    """
    Cut video from start_frame to end_frame and save as a new file.
    
    Parameters:
    - input_path: Path to the input video file.
    - output_path: Path to save the cut video.
    - start_frame: Start frame number.
    - end_frame: End frame number.
    """
    cap = load_video(input_path)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out = setup_video_writer(output_path, frame_width, frame_height)
    
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    
    for frame_num in range(start_frame, end_frame):
        ret, frame = cap.read()
        if not ret:
            break
        out.write(frame)
    
    cap.release()
    out.release()

# Process Single Video with Time
Implement function to cut a single video using start and end times in seconds.

In [3]:
# Function to cut a single video using start and end times in seconds
def cut_video_by_time(input_path, output_path, start_time, end_time):
    """
    Cut video from start_time to end_time and save as a new file.
    
    Parameters:
    - input_path: Path to the input video file.
    - output_path: Path to save the cut video.
    - start_time: Start time in seconds.
    - end_time: End time in seconds.
    """
    cap = load_video(input_path)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out = setup_video_writer(output_path, frame_width, frame_height)
    
    start_frame = int(start_time * fps)
    end_frame = int(end_time * fps)
    
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    
    for frame_num in range(start_frame, end_frame):
        ret, frame = cap.read()
        if not ret:
            break
        out.write(frame)
    
    cap.release()
    out.release()

# Process Single Video with Frames
Implement function to cut a single video using start and end frame numbers.

In [4]:
# Process Single Video with Frames

def cut_video_by_frame(input_path, output_path, start_frame, end_frame):
    """
    Cut video from start_frame to end_frame and save as a new file.
    
    Parameters:
    - input_path: Path to the input video file.
    - output_path: Path to save the cut video.
    - start_frame: Start frame number.
    - end_frame: End frame number.
    """
    cap = load_video(input_path)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out = setup_video_writer(output_path, frame_width, frame_height)
    
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    
    for frame_num in range(start_frame, end_frame):
        ret, frame = cap.read()
        if not ret:
            break
        out.write(frame)
    
    cap.release()
    out.release()

# Batch Process Multiple Videos
Create a function to process multiple videos from the inputs/ directory with specified cutting parameters.

In [5]:
# Batch Process Multiple Videos

import glob

def batch_process_videos(input_dir, output_dir, cut_params):
    """
    Batch process multiple videos from the input directory with specified cutting parameters.
    
    Parameters:
    - input_dir: Directory containing input video files.
    - output_dir: Directory to save the cut video files.
    - cut_params: List of dictionaries with keys 'filename', 'start_time', 'end_time' or 'start_frame', 'end_frame'.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    video_files = glob.glob(os.path.join(input_dir, '*.MP4'))
    
    for video_file in video_files:
        filename = os.path.basename(video_file)
        params = next((item for item in cut_params if item['filename'] == filename), None)
        
        if params:
            if output_dir is not None:
                output_path = os.path.join(output_dir, filename.split('.')[0] + '_cut.MP4')
            else:
                output_path = os.path.join(input_dir, filename.split('.')[0] + '_cut.MP4') 

            # Normalize/derive parameters
            start_time = params.get('start_time')
            end_time = params.get('end_time')
            start_frame = params.get('start_frame')
            end_frame = params.get('end_frame')

            # Derive missing values using global fps
            if start_time is not None and start_frame is None:
                start_frame = int(round(start_time * fps))
            if end_time is not None and end_frame is None:
                end_frame = int(round(end_time * fps))
            if start_frame is not None and start_time is None:
                start_time = start_frame / fps
            if end_frame is not None and end_time is None:
                end_time = end_frame / fps

            # Perform the cut based on what was provided
            if 'start_time' in params and 'end_time' in params:
                cut_video_by_time(video_file, output_path, params['start_time'], params['end_time'])
            elif 'start_frame' in params and 'end_frame' in params:
                cut_video_by_frame(video_file, output_path, params['start_frame'], params['end_frame'])
            else:
                # Mixed/derived input; use normalized values
                if start_time is not None and end_time is not None:
                    cut_video_by_time(video_file, output_path, start_time, end_time)
                elif start_frame is not None and end_frame is not None:
                    cut_video_by_frame(video_file, output_path, start_frame, end_frame)

            print(f"Video: {filename} processed.")

    if params:

        # Normalize/derive parameters
        start_time = params.get('start_time')
        end_time = params.get('end_time')
        start_frame = params.get('start_frame')
        end_frame = params.get('end_frame')

        # Write params txt in the output folder
        meta_txt_path = os.path.join(output_dir, 'params.txt')
        try:
            with open(meta_txt_path, 'w', encoding='utf-8') as f:
                f.write(f"start_time: {start_time}\n")
                f.write(f"end_time: {end_time}\n")
                f.write(f"start_frame: {start_frame}\n")
                f.write(f"end_frame: {end_frame}\n")
        except Exception as e:
            print(f"Failed to write params txt for {filename}: {e}")

    

start_time = 980
end_time = 990
# Example usage
cut_params = [
    {'filename': 'gopro1_synced.MP4', 'start_time': start_time, 'end_time': end_time},
    {'filename': 'gopro2_synced.MP4', 'start_time': start_time, 'end_time': end_time},
    {'filename': 'gopro3_synced.MP4', 'start_time': start_time, 'end_time': end_time},
    {'filename': 'gopro4_synced.MP4', 'start_time': start_time, 'end_time': end_time},
    {'filename': 'gopro5_synced.MP4', 'start_time': start_time, 'end_time': end_time},
    {'filename': 'gopro6_synced.MP4', 'start_time': start_time, 'end_time': end_time},
    {'filename': 'gopro7_synced.MP4', 'start_time': start_time, 'end_time': end_time},
    {'filename': 'gopro8_synced.MP4', 'start_time': start_time, 'end_time': end_time},
    {'filename': 'gopro9_synced.MP4', 'start_time': start_time, 'end_time': end_time},
    {'filename': 'gopro10_synced.MP4', 'start_time': start_time, 'end_time': end_time},
    {'filename': 'gopro11_synced.MP4', 'start_time': start_time, 'end_time': end_time},
    {'filename': 'gopro12_synced.MP4', 'start_time': start_time, 'end_time': end_time}
]

batch_process_videos('inputs/cha/synced/', 'inputs/cha/cha7', cut_params)

Video: gopro10_synced.MP4 processed.
Video: gopro11_synced.MP4 processed.
Video: gopro12_synced.MP4 processed.
Video: gopro1_synced.MP4 processed.
Video: gopro2_synced.MP4 processed.
Video: gopro3_synced.MP4 processed.
Video: gopro4_synced.MP4 processed.
Video: gopro5_synced.MP4 processed.
Video: gopro6_synced.MP4 processed.
Video: gopro7_synced.MP4 processed.
Video: gopro8_synced.MP4 processed.
Video: gopro9_synced.MP4 processed.
